In [1]:
from utils.bias_pipeline import BiasPipeline
pipeline = BiasPipeline()

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
04/17/2026 20:09:27 - INFO - 	 missing_keys: []
04/17/2026 20:09:27 - INFO - 	 unexpected_keys: []
04/17/2026 20:09:27 - INFO - 	 mismatched_keys: []
04/17/2026 20:09:27 - INFO - 	 error_msgs: []
04/17/2026 20:09:27 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [125]:
text = pipeline.from_json_basil("DATA/BASIL-main/articles/2017/47a209b0-fc26-4a0a-a1fe-e4310d1d2419_3.json")
text

'WASHINGTON ― In recent years, the opposing party approached Supreme Court nomination hearings by honing in on the nominee’s character and temperament or on one particular case that could possibly derail the nominee. Nominees have, in turn, generally refused to answer any question directly, in an appparent fear of revealing any bias about a potential future case they may hear. The four days of Senate Judiciary Committee hearings held this past week for President Donald Trump’s nominee to fill the late Justice Antonin Scalia’s seat, Judge Neil Gorsuch, were different. Democrats on the Senate Judiciary Committee took a different tack. They made an argument against the Supreme Court itself and its false presentation as a body above politics and power. That judges do not, as Chief Justice John Roberts once famously said, only call “balls and strikes.” At the base of this argument was a simple question that has been dogging Democrats ever since the infamous Bush v. Gore decision on the 2000

In [126]:
doc = pipeline.NER_nlp(text)
people = [(ent.start_char, ent.end_char, ent.text) for ent in doc.ents if ent.label_ == "PERSON"]
people

[(466, 478, 'Donald Trump'),
 (514, 528, 'Antonin Scalia'),
 (543, 555, 'Neil Gorsuch'),
 (795, 807, 'John Roberts'),
 (966, 970, 'Bush'),
 (974, 978, 'Gore'),
 (1149, 1156, 'Gorsuch'),
 (1208, 1215, 'Gorsuch'),
 (1551, 1569, 'Sheldon Whitehouse'),
 (1670, 1677, 'Roberts'),
 (2029, 2039, 'Whitehouse'),
 (2048, 2055, 'Gorsuch'),
 (2351, 2366, 'Zephyr Teachout'),
 (2642, 2649, 'Roberts'),
 (2744, 2752, 'Teachout'),
 (2851, 2858, 'Gorsuch'),
 (2891, 2905, 'George W. Bush'),
 (3227, 3234, 'Gorsuch'),
 (3240, 3250, 'Whitehouse'),
 (3359, 3371, 'Barack Obama'),
 (3392, 3407, 'Merrick Garland'),
 (3416, 3422, 'Scalia'),
 (3470, 3477, 'Gorsuch'),
 (3557, 3564, 'Garland'),
 (3607, 3614, 'Gorsuch'),
 (3687, 3694, 'Garland'),
 (3701, 3710, 'Pat Leahy'),
 (4001, 4008, 'Gorsuch'),
 (4071, 4081, 'Al Franken'),
 (4229, 4244, 'Merrick Garland'),
 (4325, 4332, 'Roberts'),
 (4663, 4669, 'Holder'),
 (4948, 4960, 'Mazie Hirono'),
 (4980, 4987, 'Roberts'),
 (5378, 5384, 'Scalia'),
 (5508, 5515, 'Gorsuch'),

In [127]:
def create_init_dict(doc, init_dict, secondary_data):
    for linked_ent in doc._.linkedEntities:
        #Bool if PERSON
        span = linked_ent.get_span()
        is_person = any(start <= span.start_char and span.end_char <= end for start, end, _ in people)
        if is_person:
            id = linked_ent.get_id()
            name = linked_ent.get_label()
            person_span = (span.start_char, span.end_char)
            
            #filter last name to seconday data
            if len(name.split()) == 1:
                if id not in secondary_data:
                    secondary_data[linked_ent.get_id()] = {
                        "name": name,
                        "span": [person_span],
                        "description": linked_ent.description,
                    }
                else:
                    secondary_data[linked_ent.get_id()]["span"].append(person_span)
                
            
            elif id not in init_dict:
                init_dict[linked_ent.get_id()] ={
                        "name": name,
                        "span": [person_span],
                        "description": linked_ent.description,
                        "stance" : [] 
                        }
            else:
                init_dict[linked_ent.get_id()]["span"].append(person_span)

#reallocate lastname to match subjects
def reallocate_lastnames( init_dict: dict , secondary_data: dict):
    lastnames = [ (subjects["name"].split()[-1], key)   for key , subjects in init_dict.items()]
    for subject in secondary_data.values():
        for lastname, key in lastnames:
            if subject["name"] == lastname:
                init_dict[key]["span"].extend( subject["span"])
                break
        
#resort dictionary to be most mentions
def resort_data_most_mentinos(init_dict: dict) -> dict:
    new_data = {}
    key_mentions = [ (key, len(subject["span"]))for key, subject in init_dict.items()]
    key_mentions.sort(key=lambda x: x[1], reverse=True)
    for key, _ in key_mentions:
        new_data[key] = init_dict[key]
    return new_data

def create_LE_dict(doc) -> dict:
    init_dict = {}
    secondary_data = {}
    
    create_init_dict(doc, init_dict, secondary_data)
    reallocate_lastnames(init_dict, secondary_data)
    data = resort_data_most_mentinos(init_dict)
    return data


In [128]:
data = create_LE_dict(doc)
data

{15488345: {'name': 'Neil Gorsuch',
  'span': [(543, 555),
   (1149, 1156),
   (1208, 1215),
   (2048, 2055),
   (2851, 2858),
   (3227, 3234),
   (3470, 3477),
   (3607, 3614),
   (4001, 4008),
   (5508, 5515),
   (6608, 6615)],
  'description': 'Associate Justice of the Supreme Court of the United States',
  'stance': []},
 11153: {'name': 'John Roberts',
  'span': [(795, 807), (1670, 1677), (2642, 2649), (4325, 4332), (4980, 4987)],
  'description': 'Chief Justice of the United States',
  'stance': []},
 652066: {'name': 'Sheldon Whitehouse',
  'span': [(1551, 1569), (2029, 2039), (3240, 3250), (6162, 6172)],
  'description': 'United States Senator from Rhode Island',
  'stance': []},
 1922011: {'name': 'Merrick Garland',
  'span': [(3392, 3407), (4229, 4244), (3557, 3564), (3687, 3694)],
  'description': 'American judge',
  'stance': []},
 11156: {'name': 'Antonin Scalia',
  'span': [(514, 528), (3416, 3422), (5378, 5384)],
  'description': 'former Associate Justice of the Supreme 

In [ ]:
#Run coref model
preds = pipeline.Coref_model.predict(text)
spans = preds.get_clusters(as_strings=False)



04/19/2026 17:01:12 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 24.84 examples/s]
04/19/2026 17:01:12 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  3.76it/s]


In [130]:
spans[6]


[(456, 556),
 (1149, 1156),
 (1161, 1164),
 (1195, 1215),
 (1921, 1924),
 (1962, 1965),
 (2048, 2055),
 (3047, 3050),
 (3063, 3065),
 (3078, 3080),
 (3136, 3137),
 (3227, 3234),
 (3470, 3479),
 (3607, 3616),
 (4001, 4010),
 (4118, 4121),
 (5508, 5515),
 (5645, 5648),
 (6403, 6410),
 (6608, 6615),
 (6647, 6650)]

In [131]:
cluster = preds.get_clusters()
cluster[6]

['President Donald Trump’s nominee to fill the late Justice Antonin Scalia’s seat, Judge Neil Gorsuch,',
 'Gorsuch',
 'his',
 'The polished Gorsuch',
 'you',
 'you',
 'Gorsuch',
 'his',
 'he',
 'he',
 'I',
 'Gorsuch',
 'Gorsuch’s',
 'Gorsuch’s',
 'Gorsuch’s',
 'you',
 'Gorsuch',
 'him',
 'Gorsuch',
 'Gorsuch',
 'his']

In [132]:
#update dictionary
#using span list then popping
for key , person in data.items():
    for span in person["span"]:
        person_start, person_end = span
        for i , span_list in enumerate(spans):
            in_list = any(start <= person_start and person_end <= end for start, end in span_list)
            if in_list:
                data[key]['span'] = span_list
                spans.pop(i)
                break
        break
data

{15488345: {'name': 'Neil Gorsuch',
  'span': [(456, 556),
   (1149, 1156),
   (1161, 1164),
   (1195, 1215),
   (1921, 1924),
   (1962, 1965),
   (2048, 2055),
   (3047, 3050),
   (3063, 3065),
   (3078, 3080),
   (3136, 3137),
   (3227, 3234),
   (3470, 3479),
   (3607, 3616),
   (4001, 4010),
   (4118, 4121),
   (5508, 5515),
   (5645, 5648),
   (6403, 6410),
   (6608, 6615),
   (6647, 6650)],
  'description': 'Associate Justice of the Supreme Court of the United States',
  'stance': []},
 11153: {'name': 'John Roberts',
  'span': [(781, 807),
   (1670, 1678),
   (2642, 2650),
   (4325, 4332),
   (4383, 4386),
   (4423, 4425),
   (4980, 4988),
   (5221, 5224)],
  'description': 'Chief Justice of the United States',
  'stance': []},
 652066: {'name': 'Sheldon Whitehouse',
  'span': [(1546, 1578),
   (1617, 1619),
   (1878, 1879),
   (1942, 1943),
   (1950, 1951),
   (2029, 2039),
   (2589, 2599),
   (3159, 3162),
   (3240, 3250),
   (6162, 6172)],
  'description': 'United States Sena

In [138]:
#create sorted list to replace subjects
#technally we dont need the name if we have the wikiid key
span_name = []
for key ,person in data.items():
    span_name.extend([ (start, end, person["name"], key) for start, end in person['span']])
span_name.sort( key=lambda x:x[0])
span_name

[(456, 556, 'Neil Gorsuch', 15488345),
 (497, 530, 'Antonin Scalia', 11156),
 (781, 807, 'John Roberts', 11153),
 (892, 1021, 'The Gentlemen', 60614686),
 (1121, 1134, 'The Gentlemen', 60614686),
 (1149, 1156, 'Neil Gorsuch', 15488345),
 (1161, 1164, 'Neil Gorsuch', 15488345),
 (1195, 1215, 'Neil Gorsuch', 15488345),
 (1546, 1578, 'Sheldon Whitehouse', 652066),
 (1617, 1619, 'Sheldon Whitehouse', 652066),
 (1670, 1678, 'John Roberts', 11153),
 (1878, 1879, 'Sheldon Whitehouse', 652066),
 (1921, 1924, 'Neil Gorsuch', 15488345),
 (1942, 1943, 'Sheldon Whitehouse', 652066),
 (1950, 1951, 'Sheldon Whitehouse', 652066),
 (1962, 1965, 'Neil Gorsuch', 15488345),
 (2029, 2039, 'Sheldon Whitehouse', 652066),
 (2048, 2055, 'Neil Gorsuch', 15488345),
 (2319, 2435, 'Zephyr Teachout', 17183687),
 (2574, 2575, 'Zephyr Teachout', 17183687),
 (2589, 2599, 'Sheldon Whitehouse', 652066),
 (2642, 2650, 'John Roberts', 11153),
 (2744, 2752, 'Zephyr Teachout', 17183687),
 (3047, 3050, 'Neil Gorsuch', 15488

In [139]:
#create new sentence with replaced subject and subject name and wiki key
#agian we dont need the name if we have the wiki key
#also to avoid issue we are only replacing the first mention of the subject
#since all most all sentence are Subject , verb ,object
#We only replace the first subject in the sentence and set that as the subject
result = []
for sentence in doc.sents:
    matches = [r for r in span_name if sentence.start_char <= r[0] and r[1] <= sentence.end_char]
    if matches:
        matches.sort(key=lambda x:x[0], reverse=True)
        new = sentence.text
        probable_subject = []
        wiki_id = []
        for start, end, replace_txt, wikiid in matches:
            start_ = start - sentence.start_char
            end_ = end - sentence.start_char
            new = new[:start_] + replace_txt + new[end_:]
            if replace_txt not in probable_subject:
                probable_subject.append(replace_txt)
                wiki_id.append(wikiid)

        # _,_,probable_subject, wikiid = matches[-1]
        result.append([new, probable_subject, wiki_id])
        
    else:
        continue


result  

[['The four days of Senate Judiciary Committee hearings held this past week for Neil Gorsuch',
  ['Antonin Scalia', 'Neil Gorsuch'],
  [11156, 15488345]],
 ['That judges do not, as John Roberts once famously said, only call “balls and strikes.”',
  ['John Roberts'],
  [11153]],
 ['At the base of this argument was The Gentlemen: Is the Supreme Court rigged to benefit corporations, the wealthy and, well, Republicans?',
  ['The Gentlemen'],
  [60614686]],
 ['Pressing The Gentlemen did not knock Neil Gorsuch off Neil Gorsuch steady stream of non-answers.',
  ['Neil Gorsuch', 'The Gentlemen'],
  [15488345, 60614686]],
 ['Neil Gorsuch certainly did not make any deluded or deranged statements, as many have come to expect from the president, that could lead a Republican to flip on his nomination.',
  ['Neil Gorsuch'],
  [15488345]],
 ['Sheldon Whitehouse made this point abundantly clear when Sheldon Whitehouse pointed to the series of 5-4 decisions made under John Roberts leadership that have 

determine the subject   
detemine the stance 
add to dictionary using wiki id 

wiki_id {
    name:
    decription:
    wiki_url: ?
    span:
    stances:
    [{
        sentence: [str]
        stance: str
        stance_confidence: float
        against: float
        favor: float
        neutral: float
    }
    ]
    
}

to json

data.foreach( e = >{
    
} )
    

In [140]:
def get_stance(sentence, target_name):
    labels = ["in favor of", "against", "neutral toward"]
    
    template = f"The author of this text is {{}} {target_name}."
    
    # Run the inference
    result = pipeline.stance_classifier(
        sentence, 
        candidate_labels=labels, 
        hypothesis_template=template
    )
    
    # Returns a dictionary with the labels and their confidence scores
    return result


final_data = {}

def get_Final_stance_dictionary(subject_dictionary, sentence_subject_wikiID):
    
    for sentence, subject, wiki_id in sentence_subject_wikiID:
        for name, id in zip(subject, wiki_id):
            stance_result = get_stance(sentence, name)
            subject_dictionary[id]['stance'].append(stance_result)
                

get_Final_stance_dictionary(data, result)

In [141]:
data

{15488345: {'name': 'Neil Gorsuch',
  'span': [(456, 556),
   (1149, 1156),
   (1161, 1164),
   (1195, 1215),
   (1921, 1924),
   (1962, 1965),
   (2048, 2055),
   (3047, 3050),
   (3063, 3065),
   (3078, 3080),
   (3136, 3137),
   (3227, 3234),
   (3470, 3479),
   (3607, 3616),
   (4001, 4010),
   (4118, 4121),
   (5508, 5515),
   (5645, 5648),
   (6403, 6410),
   (6608, 6615),
   (6647, 6650)],
  'description': 'Associate Justice of the Supreme Court of the United States',
  'stance': [{'sequence': 'The four days of Senate Judiciary Committee hearings held this past week for Neil Gorsuch',
    'labels': ['neutral toward', 'in favor of', 'against'],
    'scores': [0.909009575843811, 0.06101890653371811, 0.029971478506922722]},
   {'sequence': 'Pressing The Gentlemen did not knock Neil Gorsuch off Neil Gorsuch steady stream of non-answers.',
    'labels': ['neutral toward', 'against', 'in favor of'],
    'scores': [0.670904278755188, 0.27831220626831055, 0.05078350752592087]},
   {'seq

In [142]:
import json
json_string = json.dumps(data)

print(json_string) 

{"15488345": {"name": "Neil Gorsuch", "span": [[456, 556], [1149, 1156], [1161, 1164], [1195, 1215], [1921, 1924], [1962, 1965], [2048, 2055], [3047, 3050], [3063, 3065], [3078, 3080], [3136, 3137], [3227, 3234], [3470, 3479], [3607, 3616], [4001, 4010], [4118, 4121], [5508, 5515], [5645, 5648], [6403, 6410], [6608, 6615], [6647, 6650]], "description": "Associate Justice of the Supreme Court of the United States", "stance": [{"sequence": "The four days of Senate Judiciary Committee hearings held this past week for Neil Gorsuch", "labels": ["neutral toward", "in favor of", "against"], "scores": [0.909009575843811, 0.06101890653371811, 0.029971478506922722]}, {"sequence": "Pressing The Gentlemen did not knock Neil Gorsuch off Neil Gorsuch steady stream of non-answers.", "labels": ["neutral toward", "against", "in favor of"], "scores": [0.670904278755188, 0.27831220626831055, 0.05078350752592087]}, {"sequence": "Pressing The Gentlemen did not knock Neil Gorsuch off Neil Gorsuch steady str